# -----------------------------------------------
# Natural Language Processing - Assignment-1
# -----------------------------------------------

In [2]:
import re
import urllib.request
from bs4 import BeautifulSoup

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")
nltk.download("averaged_perceptron_tagger")
nltk.download("maxent_ne_chunker")
nltk.download("words")
nltk.download("wordnet")
nltk.download("omw-1.4")




[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

# ---------------------------------
# P1T1 — HTML to Plain Text Utility
# ---------------------------------

In [ ]:

def fetch_clean_text(url: str) -> str:
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_0) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15"
    }
    req = urllib.request.Request(url, headers=headers)
    html = urllib.request.urlopen(req).read()
    soup = BeautifulSoup(html, "html.parser")

    text = soup.get_text(separator=" ")

    # compress extra spaces and newlines
    text = " ".join(text.split())

    return text

url = "https://www.fccollege.edu.pk/"

# RAW HTML
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 13_0) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15"
}
req = urllib.request.Request(url, headers=headers)
raw_html = urllib.request.urlopen(req).read().decode("utf-8", errors="ignore")
print("RAW HTML (first 500 chars):\n", raw_html[:500])

# CLEAN TEXT
clean_text = fetch_clean_text(url)
print("\nCLEAN TEXT (first 500 chars):\n", clean_text[:500])

RAW HTML (first 500 chars):
 <!DOCTYPE html>
<html class="avada-html-layout-wide avada-html-header-position-top avada-is-100-percent-template" lang="en-US" prefix="og: http://ogp.me/ns# fb: http://ogp.me/ns/fb#">
<head>
	<meta http-equiv="X-UA-Compatible" content="IE=edge" />
	<meta http-equiv="Content-Type" content="text/html; charset=utf-8"/>
	<meta name="viewport" content="width=device-width, initial-scale=1" />
	<meta name='robots' content='index, follow, max-image-preview:large, max-snippet:-1, max-video-preview:-1' />

CLEAN TEXT (first 500 chars):
 Welcome to Forman Christian College | FCCU Skip to content Departments/Offices Academics School of Management Faculty of Computer and Mathematical Sciences Computer Science Mathematics Statistics Faculty of Education Education Health and Physical Education Faculty of Humanities English Mass Communication Philosophy Religious Studies Urdu Faculty of Natural Sciences Chemistry Physics Environmental Sciences Pharmacy Kauser Abdulla Malik

# --------------------------------------------------------
# P1T2 — Regex Extraction (Emails & Phones) + Tokenization
# --------------------------------------------------------

In [ ]:

# regex patterns (show these in your report)
EMAIL_RE = r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}"
PHONE_RE = r"(?:\+?\d{1,3}[\s-]?)?(?:\(?\d{2,4}\)?[\s-]?)?\d{3,4}[\s-]?\d{3,4}"

# IMPORTANT: extract emails/phones BEFORE tokenization
emails = sorted(set(re.findall(EMAIL_RE, clean_text)))
phones = sorted(set(re.findall(PHONE_RE, clean_text)))

print("Email regex:", EMAIL_RE)
print("Phone regex:", PHONE_RE)

print("\nUnique emails:", emails)
print("Unique phone numbers:", phones)

# tokenization (after extracting emails!)
sentences = nltk.sent_tokenize(clean_text)
tokens = nltk.word_tokenize(clean_text)

print("\nTotal sentences:", len(sentences))
print("Total tokens:", len(tokens))

Email regex: [a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}
Phone regex: (?:\+?\d{1,3}[\s-]?)?(?:\(?\d{2,4}\)?[\s-]?)?\d{3,4}[\s-]?\d{3,4}

Unique emails: []
Unique phone numbers: ['92 42) 99231581']

Total sentences: 19
Total tokens: 1051


# --------------------------------------------------------
# P1T3 — Text Normalization Pipeline
# --------------------------------------------------------

In [ ]:
import textwrap

def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()     # remove extra whitespace
    text = re.sub(r"\d+", "<NUM>", text)         # replace digits
    return text

before_snip = clean_text[:300]   # smaller for better screenshot
norm_text = normalize_text(clean_text)
after_snip = norm_text[:250]



print("\nBEFORE NORMALIZATION (first 300 chars):\n")
print(textwrap.fill(before_snip, width=100))

print("\n" + "-"*70)

print("\nAFTER NORMALIZATION (first 300 chars):\n")
print(textwrap.fill(after_snip, width=100))

print("\n" + "="*70)


BEFORE NORMALIZATION (first 300 chars):

Welcome to Forman Christian College | FCCU Skip to content Departments/Offices Academics School of
Management Faculty of Computer and Mathematical Sciences Computer Science Mathematics Statistics
Faculty of Education Education Health and Physical Education Faculty of Humanities English Mass
Communic

----------------------------------------------------------------------

AFTER NORMALIZATION (first 300 chars):

welcome to forman christian college | fccu skip to content departments/offices academics school of
management faculty of computer and mathematical sciences computer science mathematics statistics
faculty of education education health and physical edu



# # ------------------------------------------------------
# P1T4 — Stopwords + Corpus Statistics + Top Frequencies
# --------------------------------------------------------

In [ ]:
from collections import Counter
from nltk.corpus import stopwords

tokens_norm = nltk.word_tokenize(norm_text)

def stats(tokens_list):
    total = len(tokens_list)
    vocab = len(set(tokens_list))
    ttr = vocab / total if total else 0
    return total, vocab, ttr

# top tokens before stopword removal
top_before = Counter(tokens_norm).most_common(20)

# remove stopwords
sw = set(stopwords.words("english"))
tokens_no_sw = [t for t in tokens_norm if t.isalpha() and t not in sw]

top_after = Counter(tokens_no_sw).most_common(20)

# stats
tb, vb, ttrb = stats(tokens_norm)
ta, va, ttra = stats(tokens_no_sw)


# --------- CLEAN OUTPUT BLOCK ---------
print("="*70)
print("P1T4 - STOPWORDS REMOVAL & CORPUS STATISTICS")
print("="*70)

print("\nTop-20 BEFORE Stopword Removal:\n")
for word, freq in top_before:
    print(f"{word:<15} {freq}")

print("\n" + "-"*70)

print("\nTop-20 AFTER Stopword Removal:\n")
for word, freq in top_after:
    print(f"{word:<15} {freq}")

print("\n" + "-"*70)

print("\nCORPUS STATISTICS:\n")
print(f"{'Metric':<30}{'Before':<15}{'After'}")
print("-"*60)
print(f"{'Total Tokens':<30}{tb:<15}{ta}")
print(f"{'Vocabulary Size':<30}{vb:<15}{va}")
print(f"{'Type-Token Ratio (TTR)':<30}{round(ttrb,4):<15}{round(ttra,4)}")

print("\n" + "="*70)





P1T4 - STOPWORDS REMOVAL & CORPUS STATISTICS

Top-20 BEFORE Stopword Removal:

>               51
<               47
NUM             47
of              29
education       23
and             20
faculty         19
student         18
.               18
to              17
center          17
,               16
office          15
the             15
financial       14
fccu            12
university      11
forman          10
sciences        10
academic        10

----------------------------------------------------------------------

Top-20 AFTER Stopword Removal:

NUM             47
education       23
faculty         19
student         18
center          17
office          15
financial       14
fccu            12
university      11
forman          10
sciences        10
academic        10
aid             10
rector          9
students        9
services        8
college         7
health          7
alumni          7
programs        7

--------------------------------------------------------------

# --------------------------------------------------------
# P1T5 — POS Tagging + POS Distribution + Noun/Verb Examples
# --------------------------------------------------------

In [ ]:
from collections import Counter

pos_tags = nltk.pos_tag(tokens_no_sw)

# Count POS tags
pos_counts = Counter([tag for _, tag in pos_tags])

# Extract nouns and verbs
nouns = [w for w, t in pos_tags if t.startswith("NN")]
verbs = [w for w, t in pos_tags if t.startswith("VB")]

# -------- CLEAN OUTPUT BLOCK --------
print("="*70)
print("P1T5 - POS TAGGING & POS DISTRIBUTION")
print("="*70)

print("\nTop 10 POS Tags:\n")
for tag, count in pos_counts.most_common(10):
    print(f"{tag:<10} {count}")

print("\n" + "-"*70)

print("\nExample Nouns (15):\n")
print(nouns[:15])

print("\nExample Verbs (15):\n")
print(verbs[:15])

print("\n" + "-"*70)

print("\nFull POS Distribution:\n")
for tag, count in pos_counts.most_common():
    print(f"{tag:<10} {count}")

print("\n" + "="*70)

P1T5 - POS TAGGING & POS DISTRIBUTION

Top 10 POS Tags:

NN         334
JJ         137
NNS        123
VBP        51
NNP        47
VBG        24
VBD        19
VBZ        10
RB         9
VBN        6

----------------------------------------------------------------------

Example Nouns (15):

['forman', 'college', 'fccu', 'skip', 'content', 'academics', 'school', 'management', 'faculty', 'computer', 'sciences', 'computer', 'science', 'mathematics', 'statistics']

Example Verbs (15):

['urdu', 'pharmacy', 'faculty', 'fpsc', 'counseling', 'mercy', 'writing', 'student', 'fc', 'giving', 'alumni', 'chartered', 'dating', 'offers', 'university']

----------------------------------------------------------------------

Full POS Distribution:

NN         334
JJ         137
NNS        123
VBP        51
NNP        47
VBG        24
VBD        19
VBZ        10
RB         9
VBN        6
RBR        3
JJS        3
JJR        2
VB         2
CD         2
RP         1
PRP        1



# --------------------------------------------------------
# P1T6 — Lemmatization vs Stemming (125 tokens)
# --------------------------------------------------------

In [ ]:
from nltk.stem import PorterStemmer, LancasterStemmer, WordNetLemmatizer

porter = PorterStemmer()
lanc = LancasterStemmer()
lemm = WordNetLemmatizer()

sample_125 = tokens_no_sw[:125]


# -------- CLEAN OUTPUT BLOCK --------
print("="*90)
print("P1T6 - STEMMING vs LEMMATIZATION COMPARISON")
print("="*90)

# Table header
print(f"{'Token':<20}{'Porter Stem':<20}{'Lancaster Stem':<20}{'Lemma'}")
print("-"*90)

# Print table rows
for tok in sample_125:
    print(f"{tok:<20}{porter.stem(tok):<20}{lanc.stem(tok):<20}{lemm.lemmatize(tok)}")

print("="*90)

P1T6 - STEMMING vs LEMMATIZATION COMPARISON
Token               Porter Stem         Lancaster Stem      Lemma
------------------------------------------------------------------------------------------
welcome             welcom              welcom              welcome
forman              forman              form                forman
christian           christian           christian           christian
college             colleg              colleg              college
fccu                fccu                fccu                fccu
skip                skip                skip                skip
content             content             cont                content
academics           academ              academ              academic
school              school              school              school
management          manag               man                 management
faculty             faculti             facul               faculty
computer            comput              comput       

# ---------------------------------------
# P1T9 — Named Entity Recognition (NER)
# ---------------------------------------

In [ ]:

import nltk
nltk.download("maxent_ne_chunker_tab")

nltk.download("maxent_ne_chunker")
nltk.download("words")

def get_named_entities(pos_tags):
    tree = nltk.ne_chunk(pos_tags)
    persons, orgs, gpes = set(), set(), set()

    for chunk in tree:
        if hasattr(chunk, "label"):
            label = chunk.label()
            name = " ".join([c[0] for c in chunk])
            if label == "PERSON":
                persons.add(name)
            elif label == "ORGANIZATION":
                orgs.add(name)
            elif label in ["GPE", "LOCATION"]:
                gpes.add(name)

    return persons, orgs, gpes


persons, orgs, gpes = get_named_entities(pos_tags)

# -------- CLEAN OUTPUT BLOCK --------

print("="*80)
print("P1T9 - NAMED ENTITY RECOGNITION (NER)")
print("="*80)

print(f"\nPERSON Entities (Total: {len(persons)}):")
print("-"*80)
for name in sorted(persons)[:20]:
    print(name)

print("\n" + "-"*80)

print(f"\nORGANIZATION Entities (Total: {len(orgs)}):")
print("-"*80)
for name in sorted(orgs)[:20]:
    print(name)

print("\n" + "-"*80)

print(f"\nGPE / LOCATION Entities (Total: {len(gpes)}):")
print("-"*80)
for name in sorted(gpes)[:20]:
    print(name)

print("\n" + "="*80)

[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker_tab.zip.
[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker is already up-to-date!
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


P1T9 - NAMED ENTITY RECOGNITION (NER)

PERSON Entities (Total: 0):
--------------------------------------------------------------------------------

--------------------------------------------------------------------------------

ORGANIZATION Entities (Total: 1):
--------------------------------------------------------------------------------
NUM

--------------------------------------------------------------------------------

GPE / LOCATION Entities (Total: 0):
--------------------------------------------------------------------------------



# ---------------------------------------
P2 - (LANGUAGE MODEL AND SMOOTHING)
# ---------------------------------------

In [4]:
# ==========================================================
# PART 2: 4 LMs + Generation + Perplexity
# ==========================================================

import nltk
import random
import math
from collections import Counter, defaultdict

nltk.download("brown")
nltk.download("reuters")

from nltk.corpus import brown, reuters

random.seed(42)

# ===============================
# STEP 1: LOAD + SPLIT DATA
# ===============================
brown_sents = list(brown.sents())
random.shuffle(brown_sents)

split_index = int(0.8 * len(brown_sents))
train_raw = brown_sents[:split_index]
test_in_raw = brown_sents[split_index:]
test_out_raw = list(reuters.sents())

def add_sentence_tokens(sentences):
    return [["<s>"] + [w.lower() for w in sent] + ["</s>"] for sent in sentences]

train_sents = add_sentence_tokens(train_raw)
test_sents_in = add_sentence_tokens(test_in_raw)
test_sents_out = add_sentence_tokens(test_out_raw)

print("=" * 60)
print("PART 2 - DATA SUMMARY")
print("=" * 60)
print("Training sentences:", len(train_sents))
print("In-domain test sentences:", len(test_sents_in))
print("Out-of-domain test sentences:", len(test_sents_out))
print("\nExample training sentence:")
print(train_sents[0])


# ==========================================================
# Count words in training (excluding <s> and </s>)
train_tokens_flat = [w for s in train_sents for w in s if w not in ("<s>", "</s>")]
train_counts = Counter(train_tokens_flat)

# Words seen <= 1 time become <UNK>
UNK_THRESHOLD = 1
vocab = {w for w, c in train_counts.items() if c > UNK_THRESHOLD}
vocab.update({"<s>", "</s>", "<UNK>"})

def replace_with_unk(sents, vocab):
    fixed = []
    for s in sents:
        fixed.append([w if w in vocab else "<UNK>" for w in s])
    return fixed

# Replace rare words in TRAIN
train_sents = replace_with_unk(train_sents, vocab)

# Replace OOV in TEST (based on training vocab)
test_sents_in = replace_with_unk(test_sents_in, vocab)
test_sents_out = replace_with_unk(test_sents_out, vocab)

# Training stats after UNK replacement
train_tokens = [w for s in train_sents for w in s]
unigram_counts_global = Counter(train_tokens)

print("\n" + "=" * 60)
print("PART 2 - TRAINING STATISTICS (after <UNK>)")
print("=" * 60)
print("Total training tokens:", len(train_tokens))
print("Vocabulary size:", len(unigram_counts_global))

print("\nTop 10 most common words:")
for word, count in unigram_counts_global.most_common(10):
    print(f"{word:<15} {count}")

# ===============================
# PROVIDED-STYLE: generateSentencesToFile
# ===============================
def generateSentencesToFile(model, filename, n=20, seed=42):
    random.seed(seed)
    with open(filename, "w", encoding="utf-8") as f:
        for _ in range(n):
            sent = model.generateSentence()
            f.write(" ".join(sent) + "\n")

# ===============================
# IMPORTANT FIX #2: Perplexity in log-space
# ===============================
def perplexity(model, sents):
    total_logprob = 0.0
    total_tokens = 0

    for sen in sents:
        total_tokens += (len(sen) - 1)  # exclude first <s>

        lp = model.getSentenceLogProbability(sen)
        if lp == float("-inf"):
            return float("inf")
        total_logprob += lp

    return math.exp(-total_logprob / total_tokens)

# ==========================================================
# MODELS
# ==========================================================

# ===============================
# 1) UnigramModel (UNSMOOTHED)
# ===============================
class UnigramModel:
    def __init__(self, train_sents):
        tokens = [w for s in train_sents for w in s]
        self.counts = Counter(tokens)
        self.total = len(tokens)

        # generation: do not generate <s> in middle
        self.gen_vocab = [w for w in self.counts.keys() if w != "<s>"]
        self.gen_weights = [self.counts[w] for w in self.gen_vocab]

    def wordProb(self, w):
        return self.counts[w] / self.total if self.total > 0 else 0.0

    def generateSentence(self, max_len=50):
        sent = ["<s>"]
        for _ in range(max_len):
            w = random.choices(self.gen_vocab, weights=self.gen_weights, k=1)[0]
            sent.append(w)
            if w == "</s>":
                break
        if sent[-1] != "</s>":
            sent.append("</s>")
        return sent

    def getSentenceProbability(self, sen):
        p = 1.0
        for i in range(1, len(sen)):
            pw = self.wordProb(sen[i])
            if pw == 0.0:
                return 0.0
            p *= pw
        return p

    def getSentenceLogProbability(self, sen):
        lp = 0.0
        for i in range(1, len(sen)):
            pw = self.wordProb(sen[i])
            if pw == 0.0:
                return float("-inf")
            lp += math.log(pw)
        return lp

# ==========================================
# 2) SmoothedUnigramModel (LAPLACE)
# ==========================================
class SmoothedUnigramModel:
    def __init__(self, train_sents):
        tokens = [w for s in train_sents for w in s]
        self.counts = Counter(tokens)
        self.total = len(tokens)
        self.vocab = set(tokens)
        self.V = len(self.vocab)

        self.gen_vocab = [w for w in self.vocab if w != "<s>"]
        self.gen_weights = [self.wordProb(w) for w in self.gen_vocab]

    def wordProb(self, w):
        return (self.counts[w] + 1) / (self.total + self.V)

    def generateSentence(self, max_len=50):
        sent = ["<s>"]
        for _ in range(max_len):
            w = random.choices(self.gen_vocab, weights=self.gen_weights, k=1)[0]
            sent.append(w)
            if w == "</s>":
                break
        if sent[-1] != "</s>":
            sent.append("</s>")
        return sent

    def getSentenceProbability(self, sen):
        p = 1.0
        for i in range(1, len(sen)):
            p *= self.wordProb(sen[i])
        return p

    def getSentenceLogProbability(self, sen):
        lp = 0.0
        for i in range(1, len(sen)):
            lp += math.log(self.wordProb(sen[i]))
        return lp

# ===============================
# 3) BigramModel (UNSMOOTHED)
# ===============================
class BigramModel:
    def __init__(self, train_sents):
        self.bigram_counts = Counter()
        self.context_counts = Counter()

        for s in train_sents:
            for i in range(1, len(s)):
                prev, w = s[i-1], s[i]
                self.bigram_counts[(prev, w)] += 1
                self.context_counts[prev] += 1

        # For generation: unique next words + weights
        self.next_words = defaultdict(list)
        self.next_weights = defaultdict(list)

        temp = defaultdict(Counter)  # prev -> Counter(nextword)
        for (prev, w), c in self.bigram_counts.items():
            temp[prev][w] = c

        for prev, ctr in temp.items():
            self.next_words[prev] = list(ctr.keys())
            self.next_weights[prev] = list(ctr.values())

    def bigramProb(self, prev, w):
        denom = self.context_counts[prev]
        if denom == 0:
            return 0.0
        return self.bigram_counts[(prev, w)] / denom

    def generateSentence(self, max_len=50):
        sent = ["<s>"]
        prev = "<s>"

        for _ in range(max_len):
            if prev not in self.next_words:
                sent.append("</s>")
                break
            w = random.choices(self.next_words[prev], weights=self.next_weights[prev], k=1)[0]
            sent.append(w)
            if w == "</s>":
                break
            prev = w

        if sent[-1] != "</s>":
            sent.append("</s>")
        return sent

    def getSentenceProbability(self, sen):
        p = 1.0
        for i in range(1, len(sen)):
            prev, w = sen[i-1], sen[i]
            pb = self.bigramProb(prev, w)
            if pb == 0.0:
                return 0.0
            p *= pb
        return p

    def getSentenceLogProbability(self, sen):
        lp = 0.0
        for i in range(1, len(sen)):
            prev, w = sen[i-1], sen[i]
            pb = self.bigramProb(prev, w)
            if pb == 0.0:
                return float("-inf")
            lp += math.log(pb)
        return lp

# ==================================================
# 4) SmoothedBigramModelLI (LINEAR INTERPOLATION)
# ==================================================
class SmoothedBigramModelLI:
    def __init__(self, train_sents, lamb=0.7):
        self.lamb = lamb

        # unigram stats
        tokens = [w for s in train_sents for w in s]
        self.unigram_counts = Counter(tokens)
        self.total = len(tokens)
        self.vocab = set(tokens)
        self.V = len(self.vocab)

        # bigram stats
        self.bigram_counts = Counter()
        self.context_counts = Counter()

        for s in train_sents:
            for i in range(1, len(s)):
                prev, w = s[i-1], s[i]
                self.bigram_counts[(prev, w)] += 1
                self.context_counts[prev] += 1

        # For generation: unique next words with weights (based on counts)
        temp = defaultdict(Counter)
        for (prev, w), c in self.bigram_counts.items():
            temp[prev][w] = c

        self.next_words = defaultdict(list)
        self.next_weights = defaultdict(list)
        for prev, ctr in temp.items():
            self.next_words[prev] = list(ctr.keys())
            self.next_weights[prev] = list(ctr.values())

        self.gen_vocab = [w for w in self.vocab if w != "<s>"]

    def unigramLaplace(self, w):
        return (self.unigram_counts[w] + 1) / (self.total + self.V)

    def bigramMLE(self, prev, w):
        denom = self.context_counts[prev]
        if denom == 0:
            return 0.0
        return self.bigram_counts[(prev, w)] / denom

    def interpProb(self, prev, w):
        return self.lamb * self.bigramMLE(prev, w) + (1 - self.lamb) * self.unigramLaplace(w)

    def generateSentence(self, max_len=50):
        sent = ["<s>"]
        prev = "<s>"

        for _ in range(max_len):
            # if prev context unseen, back off to full vocab
            if prev in self.next_words:
                cand = self.next_words[prev]
                weights = self.next_weights[prev]  # generation based on seen bigram counts
                w = random.choices(cand, weights=weights, k=1)[0]
            else:
                # fallback generation using smoothed unigram
                weights = [self.unigramLaplace(w) for w in self.gen_vocab]
                w = random.choices(self.gen_vocab, weights=weights, k=1)[0]

            sent.append(w)
            if w == "</s>":
                break
            prev = w

        if sent[-1] != "</s>":
            sent.append("</s>")
        return sent

    def getSentenceProbability(self, sen):
        p = 1.0
        for i in range(1, len(sen)):
            prev, w = sen[i-1], sen[i]
            p *= self.interpProb(prev, w)
        return p

    def getSentenceLogProbability(self, sen):
        lp = 0.0
        for i in range(1, len(sen)):
            prev, w = sen[i-1], sen[i]
            lp += math.log(self.interpProb(prev, w))
        return lp

# ===============================
# TRAIN MODELS
# ===============================
unigram_model = UnigramModel(train_sents)
smooth_unigram_model = SmoothedUnigramModel(train_sents)
bigram_model = BigramModel(train_sents)
smooth_bigram_li_model = SmoothedBigramModelLI(train_sents, lamb=0.7)

print("\n" + "=" * 60)
print("PART 2 - MODELS TRAINED")
print("=" * 60)
print("1) Unigram (unsmoothed)")
print("2) Unigram (Laplace)")
print("3) Bigram (unsmoothed)")
print("4) Bigram (Linear Interp)")

# ===============================
# GENERATE 20 SENTENCES TO FILES
# ===============================
generateSentencesToFile(unigram_model, "unigram_output.txt", n=20)
generateSentencesToFile(smooth_unigram_model, "smoothunigram_output.txt", n=20)
generateSentencesToFile(bigram_model, "bigram_output.txt", n=20)
generateSentencesToFile(smooth_bigram_li_model, "smoothbigramLI_output.txt", n=20)

print("\nSaved 20 generated sentences to:")
print(" - unigram_output.txt")
print(" - smoothunigram_output.txt")
print(" - bigram_output.txt")
print(" - smoothbigramLI_output.txt")

# ===============================
# PERPLEXITY RESULTS
# ===============================
print("\n" + "=" * 60)
print("PART 2 - PERPLEXITY RESULTS")
print("=" * 60)

models = {
    "Unigram (unsmoothed)": unigram_model,
    "Unigram (Laplace)": smooth_unigram_model,
    "Bigram (unsmoothed)": bigram_model,
    "Bigram (Linear Interp)": smooth_bigram_li_model
}

for name, m in models.items():
    pp_in = perplexity(m, test_sents_in)
    pp_out = perplexity(m, test_sents_out)

    in_str = f"{pp_in:.2f}" if pp_in != float("inf") else "inf"
    out_str = f"{pp_out:.2f}" if pp_out != float("inf") else "inf"
    print(f"{name:<25} | In-domain PP: {in_str} | Out-domain PP: {out_str}")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data]   Package reuters is already up-to-date!


PART 2 - DATA SUMMARY
Training sentences: 45872
In-domain test sentences: 11468
Out-of-domain test sentences: 54716

Example training sentence:
['<s>', 'he', 'let', 'her', 'tell', 'him', 'all', 'about', 'the', 'church', '.', '</s>']

PART 2 - TRAINING STATISTICS (after <UNK>)
Total training tokens: 1020759
Vocabulary size: 24757

Top 10 most common words:
the             55897
,               46676
<s>             45872
</s>            45872
.               39479
of              29092
and             23140
to              20991
<UNK>           20399
a               18659

PART 2 - MODELS TRAINED
1) Unigram (unsmoothed)
2) Unigram (Laplace)
3) Bigram (unsmoothed)
4) Bigram (Linear Interp)

Saved 20 generated sentences to:
 - unigram_output.txt
 - smoothunigram_output.txt
 - bigram_output.txt
 - smoothbigramLI_output.txt

PART 2 - PERPLEXITY RESULTS
Unigram (unsmoothed)      | In-domain PP: 725.90 | Out-domain PP: 945.00
Unigram (Laplace)         | In-domain PP: 728.50 | Out-domain PP: 9

# -----------------------------------------------
P3T1 — Byte Pair Encoding (BPE) for Urdu Word Segmentation
# -----------------------------------------------

In [7]:
from google.colab import files
uploaded = files.upload()

Saving wordlist.txt to wordlist.txt
Saving 50_nospaces.txt to 50_nospaces.txt


In [8]:
import re, unicodedata
from collections import Counter

# =========================
# 0) filenames
# =========================
WORDLIST_TXT = "wordlist.txt"
NOSPACES_TXT = "50_nospaces.txt"

# =========================
# 1) Urdu normalize + tokenizer
# =========================
def normalize_urdu(s: str) -> str:
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\u0640", "")   # tatweel
    s = s.replace("\u200c", "")   # ZWNJ
    s = s.replace("\u200d", "")   # ZWJ
    return s

def urdu_words_from_text(text: str):
    text = normalize_urdu(text)
    # tokens = sequences of Urdu letters
    return re.findall(r"[\u0600-\u06FF]+", text)

# =========================
# 2) Read inputs
# =========================
with open(WORDLIST_TXT, "r", encoding="utf-8") as f:
    word_text = f.read()

with open(NOSPACES_TXT, "r", encoding="utf-8") as f:
    sentences = [normalize_urdu(line.strip()).replace(" ", "")
                 for line in f if line.strip()]

sentences = sentences[:50]  # as required

wordlist = urdu_words_from_text(word_text)
wordlist = list(dict.fromkeys(wordlist))  # unique, keep order

print("Dictionary words:", len(wordlist))
print("Sentences:", len(sentences))

# =========================
# 3) BPE training
# =========================
def word_to_symbols(w):
    return list(w) + ["</w>"]

def get_pair_counts(vocab_counter):
    pair_counts = Counter()
    for sym_list, freq in vocab_counter.items():
        for i in range(len(sym_list) - 1):
            pair_counts[(sym_list[i], sym_list[i+1])] += freq
    return pair_counts

def merge_pair_in_vocab(vocab_counter, pair):
    a, b = pair
    merged = a + b
    new_vocab = Counter()

    for sym_list, freq in vocab_counter.items():
        new_list = []
        i = 0
        while i < len(sym_list):
            if i < len(sym_list) - 1 and sym_list[i] == a and sym_list[i+1] == b:
                new_list.append(merged)
                i += 2
            else:
                new_list.append(sym_list[i])
                i += 1
        new_vocab[tuple(new_list)] += freq

    return new_vocab

# build initial vocab from dictionary words
vocab = Counter()
for w in wordlist:
    vocab[tuple(word_to_symbols(w))] += 1

NUM_MERGES = 500  # you can change (e.g., 200/500/1000)
merges = []

for step in range(NUM_MERGES):
    pair_counts = get_pair_counts(vocab)
    if not pair_counts:
        break
    best_pair, best_count = pair_counts.most_common(1)[0]
    if best_count < 2:
        break
    vocab = merge_pair_in_vocab(vocab, best_pair)
    merges.append(best_pair)

print("BPE merges learned:", len(merges))

# =========================
# 4) Apply BPE to each no-space sentence
# =========================
def apply_bpe(sentence, merges):
    symbols = list(sentence)
    for (a, b) in merges:
        merged = a + b
        new_syms = []
        i = 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i+1] == b:
                new_syms.append(merged)
                i += 2
            else:
                new_syms.append(symbols[i])
                i += 1
        symbols = new_syms
    return symbols

segmented_lines = []
for s in sentences:
    tokens = apply_bpe(s, merges)
    segmented_lines.append(" ".join(tokens))

with open("50_segmented.txt", "w", encoding="utf-8") as f:
    for line in segmented_lines:
        f.write(line + "\n")

print("✅ Saved output: 50_segmented.txt")

# download the output
from google.colab import files
files.download("50_segmented.txt")

Dictionary words: 5691
Sentences: 50
BPE merges learned: 500
✅ Saved output: 50_segmented.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# -----------------------------------------------
P3T2 — Average Characters per Word in Urdu Text
# -----------------------------------------------

In [12]:
from google.colab import files
uploaded = files.upload()

Saving urducorpus.txt to urducorpus.txt


In [13]:
import re, unicodedata

CORPUS_FILE = "urducorpus.txt"

def normalize_urdu(s: str) -> str:
    s = unicodedata.normalize("NFC", s)
    s = s.replace("\u0640", "")
    s = s.replace("\u200c", "")
    s = s.replace("\u200d", "")
    return s

with open(CORPUS_FILE, "r", encoding="utf-8") as f:
    text = normalize_urdu(f.read())

tokens = re.findall(r"[\u0600-\u06FF]+", text)

total_tokens = len(tokens)
avg_chars = (sum(len(t) for t in tokens) / total_tokens) if total_tokens else 0

print("Total tokens:", total_tokens)
print("Average Unicode characters per token:", round(avg_chars, 4))

Total tokens: 12654
Average Unicode characters per token: 3.637


# ---------------------------------
PART 4: EMBEDDING MODELS
# ---------------------------------

In [6]:
# =======================
#  PART-4 PIPELINE
# Primary (unlabeled): cnn_dailymail  -> for P4T1–P4T5
# Secondary (labeled): ag_news        -> for P4T6
# =======================

!pip -q install datasets scikit-learn torch

import re, math, time, random
import numpy as np
from collections import Counter, defaultdict

from datasets import load_dataset
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ------------------ tokenization ------------------
def norm(s):
    return re.sub(r"\s+", " ", s.lower()).strip()

def word_tokenize(s):
    s = norm(s)
    return re.findall(r"[a-z]+|[0-9]+", s)

# -------------------------------------------
# PRIMARY (UNLABELED) for P4T1–P4T5
#  -------------------------------------------

In [7]:
# =========================================================
# PRIMARY (UNLABELED) for P4T1–P4T5 (reduced)
# =========================================================
primary_ds = load_dataset("cnn_dailymail", "3.0.0", split="train")

PRIMARY_DOCS = 4000  # reduced hard
primary_texts = [x["article"] for x in primary_ds.select(range(PRIMARY_DOCS))]

print("\nPrimary unlabeled docs:", len(primary_texts))
print("Primary Example:", primary_texts[0][:120])


Primary unlabeled docs: 4000
Primary Example: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) f



# -------------------------------------------
# SECONDARY (LABELED) for P4T6
# -------------------------------------------

In [9]:
# =========================================================
# SECONDARY (LABELED) for P4T6
# =========================================================
ag_train = load_dataset("ag_news", split="train")
ag_test  = load_dataset("ag_news", split="test")

train_texts = [x["text"] for x in ag_train]
train_labels = [x["label"] for x in ag_train]
test_texts = [x["text"] for x in ag_test]
test_labels = [x["label"] for x in ag_test]

print("\nAG News Train:", len(train_texts), "Test:", len(test_texts))
print("AG News Example:", train_texts[0][:120])


AG News Train: 120000 Test: 7600
AG News Example: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics,


In [8]:
# =========================================================
# BPE ( small merges)
# =========================================================
def bpe_train(words, num_merges=200):
    def w2sym(w): return list(w) + ["</w>"]
    vocab = Counter(tuple(w2sym(w)) for w in words)

    def pair_counts(v):
        pc = Counter()
        for syms, f in v.items():
            for i in range(len(syms)-1):
                pc[(syms[i], syms[i+1])] += f
        return pc

    def merge_vocab(v, pair):
        a,b = pair
        m = a+b
        out = Counter()
        for syms, f in v.items():
            new=[]
            i=0
            while i < len(syms):
                if i < len(syms)-1 and syms[i]==a and syms[i+1]==b:
                    new.append(m); i += 2
                else:
                    new.append(syms[i]); i += 1
            out[tuple(new)] += f
        return out

    merges=[]
    for _ in range(num_merges):
        pc = pair_counts(vocab)
        if not pc: break
        pair, cnt = pc.most_common(1)[0]
        if cnt < 2: break
        vocab = merge_vocab(vocab, pair)
        merges.append(pair)
    return merges

def bpe_apply_word(w, merges):
    syms = list(w) + ["</w>"]
    for a,b in merges:
        m = a+b
        new=[]
        i=0
        while i < len(syms):
            if i < len(syms)-1 and syms[i]==a and syms[i+1]==b:
                new.append(m); i += 2
            else:
                new.append(syms[i]); i += 1
        syms = new
    if syms and syms[-1] == "</w>":
        syms = syms[:-1]
    return syms

def bpe_tokenize_text(s, merges):
    out=[]
    for w in word_tokenize(s):
        out.extend(bpe_apply_word(w, merges))
    return out

BPE_DOCS = 1500
NUM_MERGES = 150  # reduced hard

all_words=[]
for t in primary_texts[:BPE_DOCS]:
    all_words += word_tokenize(t)

t0=time.time()
merges = bpe_train(all_words, num_merges=NUM_MERGES)
print("\nBPE merges:", len(merges), "time:", round(time.time()-t0,2), "sec")
print("BPE demo:", bpe_tokenize_text("internationalization", merges)[:15])


BPE merges: 150 time: 22.06 sec
BPE demo: ['in', 'ter', 'n', 'ati', 'on', 'al', 'i', 'z', 'ation</w>']


# -----------
# P4T1 TF-IDF
# -----------

In [10]:
# =========================================================
# Part4T1: TF-IDF scratch
# =========================================================
def build_vocab(docs, min_freq=5, max_vocab=20000):
    freq = Counter()
    for toks in docs:
        freq.update(toks)
    items = [(w,c) for w,c in freq.items() if c >= min_freq]
    items.sort(key=lambda x:x[1], reverse=True)
    items = items[:max_vocab]
    vocab = {w:i for i,(w,_) in enumerate(items)}
    return vocab, freq

def compute_df(docs, vocab):
    df = np.zeros(len(vocab), dtype=np.int64)
    for toks in docs:
        seen=set()
        for w in toks:
            if w in vocab:
                seen.add(vocab[w])
        for i in seen:
            df[i]+=1
    return df

def tfidf(docs, vocab, df):
    N = len(docs)
    idf = np.zeros(len(vocab), dtype=np.float32)
    for w,i in vocab.items():
        idf[i] = math.log((1+N)/(1+df[i])) + 1.0

    vecs=[]
    total=0
    oov=0
    for toks in docs:
        total += len(toks)
        tf = Counter()
        for w in toks:
            if w in vocab: tf[vocab[w]] += 1
            else: oov += 1
        v={}
        for i,c in tf.items():
            v[i] = c * idf[i]
        vecs.append(v)

    oov_rate = oov/total if total else 0
    return vecs, idf, oov_rate

def sparsity(vecs, V):
    nnz = sum(len(v) for v in vecs)
    total = len(vecs) * V
    return 1 - (nnz/total)

def cos_sparse(a,b):
    dot=0.0
    for k,va in a.items():
        vb=b.get(k)
        if vb is not None:
            dot += va*vb
    na = math.sqrt(sum(v*v for v in a.values()))
    nb = math.sqrt(sum(v*v for v in b.values()))
    if na==0 or nb==0: return 0.0
    return dot/(na*nb)

N_DOCS = 2500
docs_word = [word_tokenize(t) for t in primary_texts[:N_DOCS]]
docs_bpe  = [bpe_tokenize_text(t, merges) for t in primary_texts[:N_DOCS]]

v_word, _ = build_vocab(docs_word, min_freq=5, max_vocab=15000)
v_bpe,  _ = build_vocab(docs_bpe,  min_freq=5, max_vocab=15000)

df_word = compute_df(docs_word, v_word)
df_bpe  = compute_df(docs_bpe,  v_bpe)

vec_word, idf_word, oov_word = tfidf(docs_word, v_word, df_word)
vec_bpe,  idf_bpe,  oov_bpe  = tfidf(docs_bpe,  v_bpe,  df_bpe)

print("\n---- P4T1 TF-IDF (FAST) ----")
print("Vocab word:", len(v_word), "Sparsity:", round(sparsity(vec_word, len(v_word)),4), "OOV:", round(oov_word,4))
print("Vocab bpe :", len(v_bpe),  "Sparsity:", round(sparsity(vec_bpe,  len(v_bpe)),4),  "OOV:", round(oov_bpe,4))

pairs = [(random.randrange(N_DOCS), random.randrange(N_DOCS)) for _ in range(10)]
print("Avg cosine word:", round(float(np.mean([cos_sparse(vec_word[i], vec_word[j]) for i,j in pairs])),4))
print("Avg cosine bpe :", round(float(np.mean([cos_sparse(vec_bpe[i],  vec_bpe[j])  for i,j in pairs])),4))


---- P4T1 TF-IDF (FAST) ----
Vocab word: 15000 Sparsity: 0.9817 OOV: 0.0343
Vocab bpe : 186 Sparsity: 0.1067 OOV: 0.0
Avg cosine word: 0.1651
Avg cosine bpe : 0.8419


# -----------
# P4T2 PPMI
# -----------

In [11]:
# =========================================================
# Part4T2: PPMI
# =========================================================
def build_cooc(docs, vocab, window=2, max_pairs=80000):
    cooc = Counter()
    for toks in docs:
        ids = [vocab[w] for w in toks if w in vocab]
        for i,wi in enumerate(ids):
            L = max(0, i-window)
            R = min(len(ids), i+window+1)
            for j in range(L,R):
                if j==i: continue
                cooc[(wi, ids[j])] += 1
                if len(cooc) >= max_pairs:
                    return cooc
    return cooc

def ppmi(cooc, V):
    row = np.zeros(V, dtype=np.float64)
    col = np.zeros(V, dtype=np.float64)
    total = 0.0
    for (w,c),v in cooc.items():
        row[w]+=v; col[c]+=v; total+=v

    M = defaultdict(dict)
    for (w,c),v in cooc.items():
        p_wc = v/total
        p_w  = row[w]/total
        p_c  = col[c]/total
        val = math.log((p_wc+1e-12)/((p_w*p_c)+1e-12))
        if val > 0:
            M[w][c] = val
    return M

def cosine_dict(a,b):
    if not a or not b: return 0.0
    dot=0.0
    for k,va in a.items():
        vb=b.get(k)
        if vb is not None:
            dot += va*vb
    na = math.sqrt(sum(v*v for v in a.values()))
    nb = math.sqrt(sum(v*v for v in b.values()))
    if na==0 or nb==0: return 0.0
    return dot/(na*nb)

def neighbors_ppmi(word, vocab, inv, M, topn=5):
    if word not in vocab: return []
    i = vocab[word]
    sims=[]
    for j in range(len(inv)):
        if j==i: continue
        sims.append((inv[j], cosine_dict(M[i], M[j])))
    sims.sort(key=lambda x:x[1], reverse=True)
    return sims[:topn]

PPMI_DOCS = 1500
ppmi_docs = docs_word[:PPMI_DOCS]

v_ppmi, _ = build_vocab(ppmi_docs, min_freq=10, max_vocab=2000)
inv_ppmi = {i:w for w,i in v_ppmi.items()}

print("\n---- P4T2 PPMI (FAST) ----")
for k in [2,5]:
    t0=time.time()
    cooc = build_cooc(ppmi_docs, v_ppmi, window=k, max_pairs=80000)
    M = ppmi(cooc, len(v_ppmi))
    print("Window", k, "| cooc pairs:", len(cooc), "| time:", round(time.time()-t0,2), "sec")
    for w in ["president","company","war","market"]:
        print("NN", w, ":", neighbors_ppmi(w, v_ppmi, inv_ppmi, M, topn=5))


---- P4T2 PPMI (FAST) ----
Window 2 | cooc pairs: 80000 | time: 0.56 sec
NN president : [('bush', 0.17227576634236536), ('female', 0.1683851897418059), ('official', 0.15902559510532285), ('emergency', 0.15425643820511475), ('as', 0.1490643381178242)]
NN company : [('eye', 0.17440198375606483), ('400', 0.1646040478355738), ('relief', 0.15986900277834667), ('patrol', 0.14125559249284547), ('visitors', 0.12741381957976122)]
NN war : [('nations', 0.16262062571188185), ('remain', 0.16061834795030427), ('performance', 0.14286398943995354), ('suggested', 0.1355161362801475), ('paid', 0.13395123744705797)]
NN market : [('status', 0.26480727676412613), ('prices', 0.21594345185193548), ('policies', 0.19130780501662917), ('fly', 0.16862975604206532), ('stock', 0.16687140945189824)]
Window 5 | cooc pairs: 80000 | time: 0.48 sec
NN president : [('bush', 0.4102062819032669), ('vice', 0.3175552484399176), ('snow', 0.28215844181911726), ('washington', 0.250614016757678), ('becoming', 0.22570283075988

# ------------------------------------------------
# P4T3 SGNS
# ------------------------------------------------

In [12]:
# =========================================================
# Part 4T3: SGNS
# =========================================================
def make_pairs(docs, vocab, window=2, max_pairs=40000):
    out=[]
    for toks in docs:
        ids = [vocab[w] for w in toks if w in vocab]
        for i,wi in enumerate(ids):
            L=max(0,i-window); R=min(len(ids), i+window+1)
            for j in range(L,R):
                if j==i: continue
                out.append((wi, ids[j]))
                if len(out) >= max_pairs:
                    return out
    return out

def neg_probs(freq, vocab, power=0.75):
    p = np.zeros(len(vocab), dtype=np.float64)
    for w,i in vocab.items():
        p[i] = (freq[w]**power)
    p = p / p.sum()
    return p

class SGNS(nn.Module):
    def __init__(self, V, D):
        super().__init__()
        self.in_emb  = nn.Embedding(V,D)
        self.out_emb = nn.Embedding(V,D)
        nn.init.normal_(self.in_emb.weight, 0, 0.1)
        nn.init.normal_(self.out_emb.weight,0, 0.1)

    def forward(self, center, pos, neg):
        v  = self.in_emb(center)          # (B,D)
        u  = self.out_emb(pos)            # (B,D)
        un = self.out_emb(neg)            # (B,K,D)

        pos_score = (v*u).sum(1)
        neg_score = torch.einsum("bd,bkd->bk", v, un)
        loss = -(F.logsigmoid(pos_score) + F.logsigmoid(-neg_score).sum(1)).mean()
        return loss

def train_sgns(pairs, V, probs, D=50, K=5, epochs=1, lr=0.05, bs=512, device="cpu"):
    model = SGNS(V,D).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    pairs = np.array(pairs, dtype=np.int64)
    n=len(pairs)
    t0=time.time()
    for ep in range(epochs):
        np.random.shuffle(pairs)
        run=0.0; steps=0
        for i in range(0,n,bs):
            batch = pairs[i:i+bs]
            c = torch.tensor(batch[:,0], dtype=torch.long, device=device)
            p = torch.tensor(batch[:,1], dtype=torch.long, device=device)

            neg = np.random.choice(V, size=(len(batch),K), p=probs)
            neg = torch.tensor(neg, dtype=torch.long, device=device)

            opt.zero_grad()
            loss = model(c,p,neg)
            loss.backward()
            opt.step()
            run += loss.item()
            steps += 1
        print(f"Epoch {ep+1} loss={run/max(1,steps):.4f}")
    print("Train time:", round(time.time()-t0,2), "sec")
    return model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("\nDEVICE:", DEVICE)

SGNS_DOCS = 2500
sgns_docs = [word_tokenize(t) for t in primary_texts[:SGNS_DOCS]]

v_sgns, f_sgns = build_vocab(sgns_docs, min_freq=10, max_vocab=6000)
inv_sgns = {i:w for w,i in v_sgns.items()}

pairs = make_pairs(sgns_docs, v_sgns, window=2, max_pairs=40000)
probs = neg_probs(f_sgns, v_sgns)

print("\n---- P4T3 SGNS (FAST) ----")
print("Vocab:", len(v_sgns), "Pairs:", len(pairs))

model50 = train_sgns(pairs, len(v_sgns), probs, D=50, K=5, epochs=1, device=DEVICE)
W50 = model50.in_emb.weight.detach().cpu().numpy()

def cos(u,v):
    return float(np.dot(u,v)/((np.linalg.norm(u)+1e-9)*(np.linalg.norm(v)+1e-9)))

def nearest_neighbors(word, W, vocab, inv, topn=5):
    if word not in vocab: return []
    i=vocab[word]
    v=W[i]
    sims = (W@v)/((np.linalg.norm(W,axis=1)+1e-9)*(np.linalg.norm(v)+1e-9))
    idx = np.argsort(-sims)[:topn+1]
    out=[]
    for j in idx:
        if j==i: continue
        out.append((inv[j], float(sims[j])))
        if len(out)>=topn: break
    return out

for w in ["president","company","war","market"]:
    print(w, "->", nearest_neighbors(w, W50, v_sgns, inv_sgns, topn=5))

# D=100 (still tiny, to satisfy requirement)
model100 = train_sgns(pairs, len(v_sgns), probs, D=100, K=5, epochs=1, device=DEVICE)
W100 = model100.in_emb.weight.detach().cpu().numpy()


DEVICE: cpu

---- P4T3 SGNS (FAST) ----
Vocab: 6000 Pairs: 40000
Epoch 1 loss=4.1631
Train time: 0.61 sec
president -> [('shocked', 0.47616270184516907), ('denver', 0.4604296386241913), ('aol', 0.43965235352516174), ('ballet', 0.4186064302921295), ('returned', 0.41535505652427673)]
company -> [('administration', 0.4478916525840759), ('voting', 0.44179561734199524), ('announcing', 0.4357191026210785), ('routes', 0.4277070462703705), ('turned', 0.42167574167251587)]
war -> [('wide', 0.5129271745681763), ('parliament', 0.4770171046257019), ('ships', 0.47398149967193604), ('mayor', 0.47231408953666687), ('rev', 0.46343785524368286)]
market -> [('equipment', 0.49685412645339966), ('essentially', 0.49400606751441956), ('chapter', 0.4758765697479248), ('operations', 0.47005003690719604), ('harder', 0.4524463713169098)]
Epoch 1 loss=4.1643
Train time: 0.63 sec


# ------------------------------------
# P4T4 Intrinsic (Spearman) - SGNS-50
# ------------------------------------

In [13]:
# =========================================================
# P4T4 Intrinsic (Spearman) - SGNS-50
# =========================================================
def rankdata(x):
    x=np.array(x)
    order=np.argsort(x)
    ranks=np.empty(len(x), dtype=float)
    i=0
    while i<len(x):
        j=i
        while j+1<len(x) and x[order[j+1]]==x[order[i]]:
            j+=1
        r=(i+j)/2+1
        for k in range(i,j+1):
            ranks[order[k]]=r
        i=j+1
    return ranks

def spearman(a,b):
    a=np.array(a,float); b=np.array(b,float)
    ra=rankdata(a); rb=rankdata(b)
    ra=ra-ra.mean(); rb=rb-rb.mean()
    return float((ra*rb).sum()/((np.sqrt((ra**2).sum())*np.sqrt((rb**2).sum()))+1e-9))

word_pairs = [
    ("president","leader"),
    ("company","business"),
    ("market","economy"),
    ("war","conflict"),
    ("money","cash"),
    ("doctor","hospital"),
    ("police","crime"),
    ("school","education"),
    ("internet","website"),
    ("music","song"),
]
human = [9,8,8,8,7,7,7,7,7,6]

pred=[]
usable=0
for w1,w2 in word_pairs:
    if w1 in v_sgns and w2 in v_sgns:
        pred.append(cos(W50[v_sgns[w1]], W50[v_sgns[w2]]))
        usable+=1
    else:
        pred.append(0.0)

print("\n---- P4T4 Intrinsic ----")
print("Usable pairs:", usable, "/", len(word_pairs))
print("Spearman (SGNS-50):", round(spearman(human, pred), 4))


---- P4T4 Intrinsic ----
Usable pairs: 9 / 10
Spearman (SGNS-50): -0.3737


# -------------------------
# P4T5 Sentence embeddings
# -------------------------

In [14]:
# =========================================================
# P4T5 Sentence embeddings (mean + TFIDF-weighted) using SGNS-50
# Similarity evaluation uses AG labels
# =========================================================
idf_lookup = {w: float(idf_word[i]) for w,i in v_word.items()}

def sent_mean(text):
    toks = word_tokenize(text)
    vecs=[W50[v_sgns[t]] for t in toks if t in v_sgns]
    if not vecs: return np.zeros(W50.shape[1], np.float32)
    return np.mean(vecs, axis=0)

def sent_tfidf(text):
    toks = word_tokenize(text)
    vec=np.zeros(W50.shape[1], np.float32)
    ws=0.0
    for t in toks:
        if t in v_sgns:
            w = idf_lookup.get(t, 1.0)
            vec += w * W50[v_sgns[t]]
            ws += w
    if ws==0: return vec
    return vec/ws

SIM_N=600
texts=train_texts[:SIM_N]
labels=train_labels[:SIM_N]

pairs_sim=[]
y=[]
for _ in range(600):
    i=random.randrange(SIM_N)
    if random.random()<0.5:
        same=[j for j in range(SIM_N) if labels[j]==labels[i]]
        j=random.choice(same); y.append(1)
    else:
        diff=[j for j in range(SIM_N) if labels[j]!=labels[i]]
        j=random.choice(diff); y.append(0)
    pairs_sim.append((i,j))

scores_mean=[]
scores_tfidf=[]
for i,j in pairs_sim:
    scores_mean.append(cos(sent_mean(texts[i]), sent_mean(texts[j])))
    scores_tfidf.append(cos(sent_tfidf(texts[i]), sent_tfidf(texts[j])))

thr=0.3
acc_mean = accuracy_score(y, [1 if s>=thr else 0 for s in scores_mean])
acc_tfidf= accuracy_score(y, [1 if s>=thr else 0 for s in scores_tfidf])

print("\n---- P4T5 Sentence similarity ----")
print("Mean pooling accuracy:", round(acc_mean,4))
print("TFIDF-weighted accuracy:", round(acc_tfidf,4))


---- P4T5 Sentence similarity ----
Mean pooling accuracy: 0.51
TFIDF-weighted accuracy: 0.5183


# -----------------
#  P4T6 Extrinsic
# ----------------

In [15]:
# =========================================================
# P4T6 Extrinsic (AG News) - TFIDF vs sentence(mean)
# =========================================================
CLS_TR=6000
CLS_TE=1500

Xtr_tokens = [word_tokenize(t) for t in train_texts[:CLS_TR]]
ytr = np.array(train_labels[:CLS_TR])

Xte_tokens = [word_tokenize(t) for t in test_texts[:CLS_TE]]
yte = np.array(test_labels[:CLS_TE])

v_cls, _ = build_vocab(Xtr_tokens, min_freq=5, max_vocab=15000)
df_cls = compute_df(Xtr_tokens, v_cls)
tr_vecs, idf_cls, _ = tfidf(Xtr_tokens, v_cls, df_cls)
te_vecs, _, _ = tfidf(Xte_tokens, v_cls, df_cls)

def to_dense(vecs, V):
    X=np.zeros((len(vecs), V), np.float32)
    for i,v in enumerate(vecs):
        for k,val in v.items():
            X[i,k]=val
    return X

Xtr = to_dense(tr_vecs, len(v_cls))
Xte = to_dense(te_vecs, len(v_cls))

clf = LogisticRegression(max_iter=150)
clf.fit(Xtr, ytr)
pred = clf.predict(Xte)

print("\n---- P4T6 Extrinsic (FAST) ----")
print("TFIDF Accuracy:", round(accuracy_score(yte,pred),4),
      "Macro-F1:", round(f1_score(yte,pred,average='macro'),4))

# Sentence embeddings classifier (SGNS-50 mean pooled)
XtrS = np.vstack([sent_mean(t) for t in train_texts[:CLS_TR]])
XteS = np.vstack([sent_mean(t) for t in test_texts[:CLS_TE]])

clf2 = LogisticRegression(max_iter=200)
clf2.fit(XtrS, ytr)
pred2 = clf2.predict(XteS)

print("Sentence(mean) Accuracy:", round(accuracy_score(yte,pred2),4),
      "Macro-F1:", round(f1_score(yte,pred2,average='macro'),4))


---- P4T6 Extrinsic (FAST) ----
TFIDF Accuracy: 0.8307 Macro-F1: 0.8282
Sentence(mean) Accuracy: 0.4473 Macro-F1: 0.4407
